# Graph coloring to a verified quantum oracle

This tutorial reads a graph, encodes valid colors and precolors, synthesizes an oracle, and builds Grover iterations. Run from the repository root after the native setup.

In [ ]:
from pathlib import Path
import os
import numpy as np
from IPython.display import display
from logique import read_dimacs, encode_coloring
from logique.synthesis import synthesize
from logique.verification import validate, verify_small_quantum
from logique.circuits.grover import coloring_preparation, grover_circuit
from logique.benchmarks.runner import new_run_directory
from logique.benchmarks.export import export_result
ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd()))
OUT = Path(os.environ["LOGIQUE_OUTPUT"]) if "LOGIQUE_OUTPUT" in os.environ else new_run_directory()

In [ ]:
problem = read_dimacs(ROOT / "datasets/graphs/teaching/edge.col")
encoding = encode_coloring(problem)
display(problem.to_networkx().edges)
print("Input mapping:", encoding.input_mapping)
print(encoding.verilog())

## Verify the predicate and its coherent oracle

Color bits are little-endian within each vertex; vertex labels are sorted. The raw computation may use nonzero scratch. The XOR and phase wrappers restore it.

In [ ]:
result = synthesize(encoding, "xag")
print(validate(result))
print(verify_small_quantum(result))
display(result.summary())
display(result.oracle().draw("mpl"))

## Amplitude amplification

The reflection uses the same state preparation as initialization. Only the input register is amplified; compute/copy/uncompute restores the other wires.

In [ ]:
from qiskit.quantum_info import Statevector
preparation = coloring_preparation(encoding)
for iterations in (0, 1, 2):
    circuit = grover_circuit(result, preparation=preparation, iterations=iterations)
    probabilities = Statevector.from_instruction(circuit).probabilities(result.metadata["input_qubits"])
    success = sum(p for basis, p in enumerate(probabilities) if problem.is_valid(encoding.decode(basis)))
    print(iterations, "iterations:", round(success, 6))
export_result(result, OUT, "edge")

The small example checks the workflow rather than claiming a quantum advantage. Larger graphs require explicit resource and verification budgets.